In [1]:
import carla
import random
import math

In [2]:
client = carla.Client('localhost', 2000)
world = client.get_world()
print(world)

World(id=10393467981784913697)


In [3]:
print(world.get_map().name)

Carla/Maps/Town10HD_Opt


In [4]:
# remove all the layers of the map except for the road itself.
# (the road layout isn't a toggleable layer, so it stays loaded; layers only work on "_Opt" maps like Town10HD_Opt)
# Unload one layer at a time: combining them with `|` yields an int, which the API rejects.
for layer in [
    carla.MapLayer.Buildings,
    carla.MapLayer.Decals,
    carla.MapLayer.Foliage,
    carla.MapLayer.Ground,
    carla.MapLayer.ParkedVehicles,
    carla.MapLayer.Particles,
    carla.MapLayer.Props,
    carla.MapLayer.StreetLights,
    carla.MapLayer.Walls,
]:
    world.unload_map_layer(layer)

# Traffic lights, traffic signs and the remaining buildings are part of the base level, not toggleable layers,
# so hide them individually as environment objects.
ids_to_hide = set()
for label in [
    carla.CityObjectLabel.TrafficLight,
    carla.CityObjectLabel.TrafficSigns,
    carla.CityObjectLabel.Poles,  # sign / traffic light poles
    carla.CityObjectLabel.Buildings,
]:
    ids_to_hide.update(obj.id for obj in world.get_environment_objects(label))
world.enable_environment_objects(ids_to_hide, False)
print(f"Hid {len(ids_to_hide)} environment objects")

Hid 232 environment objects


In [26]:
# spawn a car at a fixed recommended spawn point (same car + location every run)
VEHICLE_TYPE = 'vehicle.tesla.model3'
SPAWN_INDEX = 5

# re-running would collide with the previous car at the same spot, so remove it first
try:
    vehicle.destroy()
except NameError:
    pass

vehicle_bp = world.get_blueprint_library().find(VEHICLE_TYPE)

spawn_points = world.get_map().get_spawn_points()
spawn_point = spawn_points[SPAWN_INDEX]

vehicle = world.spawn_actor(vehicle_bp, spawn_point)
print(f"Spawned {vehicle.type_id} (id {vehicle.id}) at spawn point {SPAWN_INDEX}: {spawn_point.location}")

# move the spectator camera above the car so it's visible
spectator = world.get_spectator()
spectator.set_transform(carla.Transform(
    spawn_point.location + carla.Location(z=30),
    carla.Rotation(pitch=-90),
))

Spawned vehicle.tesla.model3 (id 35) at spawn point 5: Location(x=-106.649590, y=-17.073978, z=0.600000)


In [12]:
# visualize waypoints
for wp in world.get_map().generate_waypoints(2.0):  # one every 2 m on all lanes
    world.debug.draw_point(wp.transform.location, size=0.1, life_time=120)

In [27]:
# start autopilot

carla_map = world.get_map()

def get_speed():
    v = vehicle.get_velocity()
    speed_ms = math.sqrt(v.x**2 + v.y**2 + v.z**2)
    return speed_ms

def get_steering_angle(x: float) -> float:
    """Return VehicleControl.steer in [-1, 1] that points the car at the leftmost lane waypoint x meters ahead."""
    waypoint = carla_map.get_waypoint(vehicle.get_location())
    
    # get waypoints x meters ahead (next() returns a list; at junctions it has several branches)
    candidates = waypoint.next(x)
    if not isinstance(candidates, (list, tuple)):
        candidates = [candidates]
    transform = vehicle.get_transform()

    def heading_error(w):
        # signed angle (deg) from the vehicle's heading to waypoint w.
        # CARLA yaw is clockwise-positive, so positive = target is to the right, negative = to the left
        loc = w.transform.location
        desired_yaw = math.degrees(math.atan2(loc.y - transform.location.y, loc.x - transform.location.x))
        error = desired_yaw - transform.rotation.yaw
        return (error + 180) % 360 - 180  # wrap to [-180, 180]

    # leftmost waypoint = the one with the smallest (most negative) signed angle
    error = min(heading_error(w) for w in candidates)
    print(waypoint.road_id, waypoint.lane_id, waypoint.is_junction, len(candidates), error)  # DEBUG
    # normalize by the front wheels' max steer angle (degrees) -> steer in [-1, 1]
    max_steer = vehicle.get_physics_control().wheels[0].max_steer_angle
    return max(-1.0, min(1.0, error / max_steer))

MAX_SPEED = 300
TARGET_SPEED = 10
LOOKAHEAD = 5.0  # meters ahead of the car to aim at
GAIN = 1.0  # scales how aggressively throttle responds to speed error
DAMP = 0.5  # damping on speed change: raise to reduce oscillation, lower for a faster rise
throttle = 0
prev_speed = get_speed()

while True:
    world.wait_for_tick()  # only update once per simulator tick

    # id controller for speed
    curr_speed = get_speed()
    throttle += GAIN * (TARGET_SPEED - curr_speed) / MAX_SPEED - DAMP * (curr_speed - prev_speed)
    prev_speed = curr_speed
    throttle = max(0, min(1, throttle))  # cap throttle to [0, 1]
    steer = get_steering_angle(LOOKAHEAD)
    vehicle.apply_control(carla.VehicleControl(throttle=throttle, steer=steer))
    print(curr_speed, steer)

    


6 -1 False 1 1.1449414345319724
0.0 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.0 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.0003328513994347392 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.0003589534609398341 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.0005854821065854412 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.000283310393710853 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.0002258658351776432 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.00022844075284180333 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.0001137685758306924 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.00011931180779382425 0.0163563079902957
6 -1 False 1 1.1449414345319724
6.702184327917203e-05 0.0163563079902957
6 -1 False 1 1.1449414345319724
0.00014627932978343187 0.0163563079902957
6 -1 False 1 1.1449414345319724
4.826426527853836e-05 0.0163563079902957
6 -1 False 1 1.1449414345319724
8.050203199722186e-05 0.0163563079902957
6 -1 False 1

KeyboardInterrupt: 